# Breast Cancer Histopathology

**Task:** Binary - IDC Positive vs IDC Negative histopathology

**Model:** densenet121

**Dataset:** [paultimothymooney/breast-histopathology-images](https://www.kaggle.com/datasets/paultimothymooney/breast-histopathology-images)


In [ ]:
import sys
sys.path.insert(0, '../..')

import torch
from shared.config import DEVICE, BATCH_SIZE, EPOCHS, LEARNING_RATE, EARLY_STOPPING_PATIENCE
from shared.models import create_model
from shared.pipelines.dataset import create_dataloaders
from shared.pipelines.train import train_model

print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')


In [ ]:
from config import PROJECT_ID, MODEL_NAME, CLASSES, IMG_SIZE
from shared.config import get_trained_model_path

checkpoint_path = str(get_trained_model_path(PROJECT_ID))
print(f'Project: {PROJECT_ID}')
print(f'Model: {MODEL_NAME}')
print(f'Classes: {CLASSES}')


In [ ]:
train_loader, val_loader, test_loader, ds_classes = create_dataloaders(
    data_root='data',
    class_names=CLASSES,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)
print(f'Train: {len(train_loader.dataset)} images')
print(f'Val: {len(val_loader.dataset)} images')
print(f'Test: {len(test_loader.dataset)} images')


In [ ]:
model = create_model(MODEL_NAME, num_classes=len(CLASSES))
total = sum(p.numel() for p in model.parameters())
print(f'Parameters: {total:,}')


In [ ]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    num_epochs=EPOCHS,
    lr=LEARNING_RATE,
    patience=EARLY_STOPPING_PATIENCE,
    checkpoint_path=checkpoint_path,
)
print('Training complete!')


In [ ]:
from shared.utils.metrics import compute_metrics

model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE, weights_only=True))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        preds = probs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

m = compute_metrics(all_labels, all_preds, all_probs)
print(f'Accuracy:  {m["accuracy"]:.4f}')
print(f'Precision: {m["precision"]:.4f}')
print(f'Recall:    {m["recall"]:.4f}')
print(f'F1-Score:  {m["f1_score"]:.4f}')
if m.get('roc_auc'):
    print(f'ROC-AUC:   {m["roc_auc"]:.4f}')


In [ ]:
from shared.explainers.xai_factory import run_all_explainers
from shared.pipelines.transforms import get_inference_transform
from PIL import Image
import os, random

test_dir = 'data/test'
if os.path.isdir(test_dir):
    class_dir = random.choice(os.listdir(test_dir))
    img_name = random.choice(os.listdir(os.path.join(test_dir, class_dir)))
    img_path = os.path.join(test_dir, class_dir, img_name)
    print(f'Sample: {img_path}')

    pil = Image.open(img_path).convert('RGB')
    transform = get_inference_transform(IMG_SIZE)
    tensor = transform(pil).unsqueeze(0).to(DEVICE)

    result = run_all_explainers(model, tensor, CLASSES, device=DEVICE)
    print(f'Prediction: {result["predictions"]["predicted_class"]}')
    print(f'Confidence: {result["predictions"]["confidence"]:.2%}')

    from IPython.display import display, HTML
    html = '<div style="display:grid;grid-template-columns:repeat(2,1fr);gap:10px;">'
    for key, exp in result['explanations'].items():
        if 'overlay_base64' in exp:
            html += f'<div style="padding:10px;border:1px solid #ddd;">'
            html += f'<h4>{exp["label"]}</h4>'
            html += f'<img src="data:image/png;base64,{exp["overlay_base64"]}" style="width:100%;"/>'
            html += '</div>'
    display(HTML(html + '</div>'))
